In [25]:
from langchain_ollama import ChatOllama
from langchain_core.tools import InjectedToolArg
from typing import Annotated
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import initialize_agent, AgentType
import requests
import json

In [8]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function retrieves the currency conversion factor between a specified base currency and a target currency.
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  Given a currency conversion rate, this function computes the target currency value from a given base currency value.
  """

  return base_currency_value * conversion_rate

In [9]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [10]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782604801,
 'time_last_update_utc': 'Sun, 28 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1782691201,
 'time_next_update_utc': 'Mon, 29 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.4376}

In [11]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [13]:
llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

In [14]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [17]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 INR to USD?')]

In [18]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 INR to USD?', additional_kwargs={}, response_metadata={})]

In [19]:
ai_message = llm_with_tools.invoke(messages)

In [20]:
messages.append(ai_message)

In [21]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'c9ebc820-5406-4f84-b007-4aecb4b5258e',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '4cadf11d-c18a-4880-98b0-ca11ae749410',
  'type': 'tool_call'}]

In [23]:
for tool_call in ai_message.tool_calls:
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)
  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [24]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 INR to USD?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-06-28T13:59:03.181188Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15286965792, 'load_duration': 6641288792, 'prompt_eval_count': 248, 'prompt_eval_duration': 647389000, 'eval_count': 79, 'eval_duration': 7927283000, 'logprobs': None, 'model_name': 'qwen2.5:3b'}, id='run--019f0e86-9133-7e62-a4d7-7220598cdfd3-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'c9ebc820-5406-4f84-b007-4aecb4b5258e', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 0.01059}, 'id': '4cadf11d-c18a-4880-98b0-ca11ae749410', 'type': 'tool_call'}], usage_metadata={'input_tokens': 248, 'output_tokens': 79, 'total_tok

In [26]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR (Indian Rupee) and USD (US Dollar) is 0.01059. Based on this conversion rate, 10 INR converts to approximately 0.1059 USD.\n\nPlease note that the exact value might have slight variations due to rounding or other factors.'

In [31]:
agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False 
)

In [33]:
response = agent_executor.invoke({"input": user_query})
answer = response["output"]

In [34]:
answer

'Hello! How can I assist you today?'